In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from tqdm.auto import tqdm
from functions import estimate_parameters, formula_15, garch_returns

In [ ]:

def formula_15_from_data_NOT_USED( r: np.ndarray | pd.Series ) -> float:
    """Variance of the Sharpe ratio, estimated from the data"""
    SR, skew, kurtosis, alpha, beta, T = estimate_parameters( r )
    return formula_15( SR, skew, kurtosis, alpha, beta, T )


In [ ]:
def f(T):
        
    df = 5
    sigma = .15
    mu = .08
    alpha = .05
    beta = .08
    # alpha = beta = 0
    # T = 100
    SR = mu / sigma
    skew = 0
    kurtosis = 3 + 6 / ( df - 4 )

    assert 1 - alpha**2 * kurtosis - 2 * alpha * beta - beta**2 > 0
    assert alpha + beta < 1

    stats = []
    for _ in range(10_000):
        ys, zs, vs = garch_returns( size = T, df = df, mu = mu, sigma = sigma, alpha = alpha, beta = beta )
        stats.append( { 
            'mean': ys.mean(),
            'std': ys.std(),
            'SR': ys.mean() / ys.std(),
            'skew': scipy.stats.skew( ys ),
            'kurtosis': 3 + scipy.stats.kurtosis( zs )
        } )

    stats = pd.DataFrame( stats )
    #display( stats.mean() )

    # Those two should be equal -- they are not
    return { 
        'T': T,
        'sigma': sigma,
        'mu': mu,
        'alpha': alpha,
        'beta': beta,
        'SR': SR,
        'sample SR': stats['SR'].mean(),
        'skew': skew,
        'kurtosis': kurtosis,
        'formula (15)': formula_15( SR, skew, kurtosis, alpha, beta, T ), 
        'sample Var[SR]': np.var( stats['SR'] ),
    }

res = pd.DataFrame( [ 
    f(T) 
    for T in tqdm([10,20,30,50,100,200,300,500,1000,2000,3000,5000,10_000] )
] )
res

In [ ]:
# The relative error tends to 1
res['sample Var[SR]'] / res['formula (15)']